# ۲. پاک‌سازی، یکسان‌سازی نام تیم‌ها و ادغام

این مرحله سه منبع را پاک‌سازی می‌کند، نام‌های متفاوت یک باشگاه را به نام جدول All-Time نگاشت می‌کند و خروجی‌های استاندارد می‌سازد.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "DataSet2").is_dir())
ANALYSIS_ROOT = ROOT / "UCL_Analysis2"
OUTPUT = ANALYSIS_ROOT / "Output"
sys.path.insert(0, str(ANALYSIS_ROOT / "src"))
pd.set_option("display.max_columns", 100)
plt.style.use("seaborn-v0_8-whitegrid")
print("Repository root:", ROOT)


In [ ]:
from ucl_analysis import build_and_save_all

audit = build_and_save_all(ROOT, OUTPUT)
pd.Series(audit, name="value").to_frame()


## منطق ادغام

فایل تاریخ‌دار همیشه اولویت دارد. برای رکوردهایی که در هر دو فایل یک فصل، دو تیم و نتیجه یکسان دارند فقط نسخه تاریخ‌دار نگه داشته می‌شود. رکوردهای واقعاً اضافه `ucl.csv` با برچسب `ucl_supplemental` افزوده می‌شوند. چون فایل دوم تاریخ ندارد، این رکوردها فقط در تحلیل تجمعی نرخ برد استفاده می‌شوند، نه Elo زمانی.

In [ ]:
alltime = pd.read_csv(OUTPUT / "alltime_team_strength.csv")
detailed = pd.read_csv(OUTPUT / "detailed_matches_clean.csv")
ucl = pd.read_csv(OUTPUT / "ucl_matches_deduplicated.csv")
combined = pd.read_csv(OUTPUT / "combined_unique_matches.csv")

summary = pd.DataFrame([
    {"dataset": "All-time teams", "rows": len(alltime), "unique_teams": alltime.Team.nunique()},
    {"dataset": "Dated matches", "rows": len(detailed), "unique_teams": len(set(detailed.home_team) | set(detailed.away_team))},
    {"dataset": "UCL deduplicated", "rows": len(ucl), "unique_teams": len(set(ucl.home_team) | set(ucl.away_team))},
    {"dataset": "Combined multiset union", "rows": len(combined), "unique_teams": len(set(combined.home_team) | set(combined.away_team))},
])
summary


In [ ]:
display(alltime[["Rank", "Team", "Matches", "calculated_points", "points_per_match", "win_rate", "alltime_elo_proxy"]].head(10))
display(detailed[["date", "season_start", "home_team", "away_team", "home_goals", "away_goals", "phase", "result"]].head())
display(combined.source.value_counts().rename("matches").to_frame())


## تعریف قدرت کلی Elo-like

جدول All-Time ترتیب زمانی مسابقات را ندارد، پس Elo واقعی از آن قابل محاسبه نیست. امتیاز `alltime_elo_proxy` نرخ نتیجه تاریخی را با یک prior بیست‌بازی به سمت ۱۵۰۰ shrink می‌کند و سپس به مقیاس Elo تبدیل می‌کند. این شاخص قدرت توصیفی است، نه Elo زمانی.

In [ ]:
columns = ["Team", "Matches", "Wins", "Draws", "Losses", "win_rate", "points_per_match", "alltime_elo_proxy"]
alltime.nlargest(20, "alltime_elo_proxy")[columns].reset_index(drop=True)
